# ITL Column Dip Along Saturated Stars (v1)

**Author:** Aaron Roodman
**Date Created:** 2026-10-15
**Last Modified:** 2026-10-15
**Status:** Draft
**Keywords:** ITL, CCD, saturation, sky background, ISR, LSSTCam

## Description

ITL CCDs show a depression of the sky level -- the "ITL dip" -- in the
columns running through a saturated star. E2V CCDs do not. This notebook
pulls low galactic latitude LSSTCam images, which are rich in saturated
stars, runs ISR by hand **with the dip masking turned off**, and extracts
column cuts so the dip can be seen and measured against the sky level.

Key functionality:
1. Find low galactic latitude visits from the Consolidated Database (ConsDB).
2. Run `IsrTaskLSST` by hand with `doITLDipMask = False` so the dip pixels
   keep their values and the sky is never subtracted.
3. Display the CCD with the stretch set around the sky level, so a
   few-count dip is visible.
4. Locate saturated stars by magnitude from the reference catalog.
5. Profile across the dip columns and measure the depth below the sky.
6. Repeat on an E2V CCD as a control.

**Output:** PNG figures in `camera/output/itl_dip/<day_obs>/`

**Dependencies:** LSST Science Pipelines (`lsst_distrib`) and standard
scientific Python only -- `numpy`, `pandas`, `matplotlib`, `scipy`,
`astropy`, `requests`. No external or personal packages, so this notebook
is self-contained and portable.

## Change Log

| Date | Author | Description |
|------|--------|-------------|
| 2026-10-15 | Aaron Roodman | Initial version |

## Why ISR is re-run here

The Science Pipelines already know about this feature and **mask it by
default**. `lsst.ip.isr.isrTaskLSST.IsrTaskLSSTConfig` has

```python
doITLDipMask = True                              # default
itlDipMaskPlanes = ["SUSPECT", "ITL_DIP"]        # default
```

and `lsst.ip.isr.isrFunctions.maskITLDip` flags the dip columns. Masking
them `SUSPECT` is right for science, but it hides exactly the pixels we
want to look at, so this notebook sets `doITLDipMask = False`.

Two further reasons we run ISR rather than read a stored product:

- `preliminary_visit_image` (PVI) is **background subtracted** -- its
  median is about -10 counts, with the sky held separately in
  `preliminary_visit_image_background`. The dip is a sky-level feature.
- `post_isr_image` is **not persisted for science exposures** in the
  nightly collections; only `bias`, `dark`, `flat` and `cwfs` frames keep
  one.

Note also that the per-amp gate `itlDipBackgroundFraction` is `0.0` in the
bare `IsrTaskLSSTConfig` defaults, which disables dip masking entirely.
The real values are set per detector in
`obs_lsst/config/lsstCam/isrLSST.py` (ticket DM-50343), which names **25
ITL detectors** with known dips and no E2V detectors at all. Those 25 are
listed in the Parameters cell below and are the best place to look.

## Table of Contents

1. [Parameters](#params)
2. [Setup & Imports](#setup)
3. [Helper Functions](#functions)
4. [Find low galactic latitude visits](#visits)
5. [Run ISR without dip masking](#isr)
6. [Display the CCD at the sky level](#display)
7. [Column cuts through saturated stars](#cuts)
8. [E2V control](#e2v)
9. [PVI cross-check](#pvi)

<a id='params'></a>
## 1. Parameters

In [1]:
# ============================================================
# Parameters -- all configurable values collected here
# ============================================================

butler_repo = '/repo/main'
raw_collection = 'LSSTCam/raw/all'
calib_collection = 'LSSTCam/calib'

# Nightly collection holding preliminary_visit_image (PVI).  The 2026 nights
# are in the bare-numbered nightlyValidation collections, not ones named by
# day_obs (those stop in 2025-09).  '68' covers day_obs 20260708-20260709.
# The PVI is used only for its WCS and photometric calibration.
pvi_collection = 'LSSTCam/runs/nightlyValidation/68'

# Reference catalog, used to locate bright stars by magnitude.  'the_monster'
# keeps the bright stars that the pipeline's own source catalog drops once
# they saturate.  monster_ComCam_* are the LSST filter fluxes, in nJy.
refcat_name = 'the_monster_20250219'
refcat_collection = 'refcats'

# Target exposure.  A 'lowdust' field, i band, 30 s: sparse enough that the
# sky between stars is clearly visible, which is what the dip must be
# measured against.  Crowded galactic-plane fields ('bulgy', 'dusty_plane')
# have plenty of saturated stars but the confusion noise between them is as
# large as the dip itself, so they are the wrong place to start.
day_obs = 20260708
seq_num = 22

# Detectors to study, from the ISR list of ITL CCDs with production dip
# rules, as name -> (itlDipMinWidth in pixels, itlDipBackgroundFraction).
# itlDipBackgroundFraction is dimensionless: the dip's flux deficit at its
# centre, scaled by width, over the sky level.  Values from
# obs_lsst/config/lsstCam/isrLSST.py (ticket DM-50343).
DIP_RULES = {
    'R01_S12': (8, 0.0030),
    'R02_S10': (6, 0.0025), 'R02_S12': (6, 0.0025), 'R10_S02': (6, 0.0025),
    'R41_S21': (6, 0.0025), 'R42_S02': (6, 0.0025), 'R43_S11': (6, 0.0025),
    'R43_S20': (6, 0.0025),
    'R01_S10': (10, 0.0015), 'R01_S11': (10, 0.0015), 'R01_S20': (10, 0.0015),
    'R10_S10': (10, 0.0015), 'R41_S00': (10, 0.0015),
    'R01_S21': (13, 0.0015), 'R03_S20': (13, 0.0015), 'R10_S22': (13, 0.0015),
    'R01_S22': (15, 0.0010), 'R02_S02': (15, 0.0010), 'R02_S11': (15, 0.0010),
    'R02_S21': (15, 0.0010), 'R03_S00': (15, 0.0010), 'R03_S02': (15, 0.0010),
    'R10_S01': (15, 0.0010), 'R41_S01': (15, 0.0010), 'R43_S01': (15, 0.0010),
}

itl_detector = 14          # R02_S12, ITL, one of the strongest rules
e2v_detector = 39          # R11_S10, E2V, the control

# Dip selection.  sat_row_fraction is the fraction of the column's rows that
# the bleed trail saturates.  A trail filling the whole column leaves no
# clean sky to measure against, so prefer partially filled columns.
max_sat_row_fraction = 0.5
min_dip_width_pix = 10
n_dips_to_plot = 3

# Profile geometry, in pixels.  The dip runs along y, so it is profiled
# ACROSS columns as a function of x, using rows beyond the end of the bleed
# trail where the sky is clean.
trail_clearance_pix = 250  # start this far beyond the last saturated row
profile_half_width_pix = 200
min_profile_rows = 300

# Star magnitude range, used to report which star produced each dip.  Stars
# of 13-15 mag saturate in a 30 s exposure (peaks of order 2e5 counts)
# without the full-column bleed trails of the very brightest.
mag_min, mag_max = 13.0, 15.0
edge_margin_pix = 500      # keep stars this far from the detector edge
n_stars_to_plot = 4

# Visit search (section 4)
max_abs_gal_b_deg = 90.0
target_names = ('lowdust',)
min_exp_time_s = 10.0
search_day_obs_min = 20260708
search_day_obs_max = 20260709

consdb_url = 'https://usdf-rsp.slac.stanford.edu/consdb/query'

# Output goes to the study's output symlink, which points at
# studies/camera/itl_dip under the S3DF data root.  Every figure lands in one
# multi-page PDF rather than scattered PNG files.
# Relative to this notebook, which lives in <study>/notebooks/.
output_dir = '../output'
pdf_name = f'itl_dip_{day_obs}_{seq_num}.pdf'

<a id='setup'></a>
## 2. Setup & Imports

In [2]:
import copy
import os
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages
import requests

import astropy.units as u
from astropy.coordinates import SkyCoord
from astropy.visualization import ZScaleInterval

from lsst.daf.butler import Butler
from lsst.obs.lsst import LsstCam
from lsst.ip.isr.isrTaskLSST import IsrTaskLSST, IsrTaskLSSTConfig
from lsst.meas.algorithms import ReferenceObjectLoader

plt.rcParams.update({
    'figure.figsize': (9, 6),
    'figure.dpi': 110,
    'image.origin': 'lower',
    'image.cmap': 'gray',
    'font.size': 10,
    'axes.grid': False,
})

output_dir = Path(output_dir).resolve()
output_dir.mkdir(parents=True, exist_ok=True)
pdf_path = output_dir / pdf_name
pdf = PdfPages(pdf_path)

butler = Butler(butler_repo)
camera = LsstCam.getCamera()
exposure_id = day_obs * 100000 + seq_num
n_detector_rows = camera[itl_detector].getBBox().getHeight()

print(f'butler repo        : {butler_repo}')
print(f'exposure_id        : {exposure_id}')
print(f'ITL detector       : {itl_detector} ({camera[itl_detector].getName()}, '
      f'{camera[itl_detector].getPhysicalType()})')
print(f'E2V detector       : {e2v_detector} ({camera[e2v_detector].getName()}, '
      f'{camera[e2v_detector].getPhysicalType()})')
print(f'output_dir         : {output_dir}')
print(f'figures            : {pdf_path}')

butler repo        : /repo/main
exposure_id        : 2026070800022
ITL detector       : 14 (R02_S12, ITL)
E2V detector       : 39 (R11_S10, E2V)
output_dir         : /sdf/data/rubin/user/roodman/LSST/rubin-work/studies/camera/itl_dip
figures            : /sdf/data/rubin/user/roodman/LSST/rubin-work/studies/camera/itl_dip/itl_dip_20260708_22.pdf


### Environment note

If `loadLSST.sh` fails with `Could not find conda environment:
lsst-scipipe-13.1.0-exact`, a stale `LSST_CONDA_ENV_NAME` is set in the
shell. Clear it before sourcing the stack:

```bash
unset LSST_CONDA_ENV_NAME
source /cvmfs/sw.lsst.eu/almalinux-x86_64/lsst_distrib/w_2026_42/loadLSST.sh
setup lsst_distrib
```

<a id='functions'></a>
## 3. Helper Functions

In [3]:
def consdb_query(sql, url=None):
    """Run a SQL query against the ConsDB REST endpoint.

    Uses the REST API with `requests` so the notebook does not depend on
    ``lsst.summit.utils``, which is not part of ``lsst_distrib``.

    Parameters
    ----------
    sql : `str`
        SQL query text.
    url : `str`, optional
        ConsDB query endpoint. Defaults to the ``consdb_url`` parameter.

    Returns
    -------
    result : `pandas.DataFrame`
        Query result.

    Notes
    -----
    The access token is read from ``~/.lsst/consdb_token`` if present, and
    otherwise from the ``ACCESS_TOKEN`` environment variable. It is never
    printed.
    """
    token_file = Path.home() / '.lsst' / 'consdb_token'
    if token_file.exists():
        token = token_file.read_text().strip()
    else:
        token = os.environ['ACCESS_TOKEN']

    response = requests.post(url or consdb_url, json={'query': sql},
                             auth=('user', token), timeout=600)
    response.raise_for_status()
    payload = response.json()
    return pd.DataFrame(payload['data'], columns=payload['columns'])


def find_low_galactic_latitude_visits(day_obs_min, day_obs_max,
                                      max_abs_b_deg=5.0,
                                      targets=None,
                                      min_exp_time=10.0):
    """Find science visits close to the galactic plane.

    ``cdb_lsstcam.visit1`` stores no galactic coordinates, so galactic
    latitude is computed from the boresight ICRS position with astropy.

    Parameters
    ----------
    day_obs_min, day_obs_max : `int`
        Inclusive ``day_obs`` range, as ``YYYYMMDD``.
    max_abs_b_deg : `float`, optional
        Keep visits with absolute galactic latitude below this, in degrees.
    targets : `tuple` [`str`], optional
        If given, keep only these ``target_name`` values. The galactic-plane
        survey fields are ``bulgy`` and ``dusty_plane``.
    min_exp_time : `float`, optional
        Minimum exposure time in seconds.

    Returns
    -------
    visits : `pandas.DataFrame`
        One row per visit, with ``gal_l`` and ``gal_b`` in degrees added,
        sorted by absolute galactic latitude.
    """
    visits = consdb_query(f"""
        SELECT visit_id, day_obs, seq_num, band, exp_time, s_ra, s_dec,
               airmass, target_name, science_program
        FROM cdb_lsstcam.visit1
        WHERE day_obs BETWEEN {day_obs_min} AND {day_obs_max}
          AND img_type = 'science'
          AND exp_time > {min_exp_time}
          AND s_ra IS NOT NULL
    """)
    coords = SkyCoord(visits.s_ra.values * u.deg,
                      visits.s_dec.values * u.deg, frame='icrs').galactic
    visits['gal_l'] = coords.l.deg
    visits['gal_b'] = coords.b.deg

    keep = np.abs(visits.gal_b) < max_abs_b_deg
    if targets is not None:
        keep &= visits.target_name.isin(targets)
    visits = visits[keep].copy()
    return visits.reindex(np.abs(visits.gal_b).sort_values().index)

In [4]:
# Calibration dataset type -> IsrTaskLSST.run() keyword.  The names differ.
CALIB_DATASET_TO_KWARG = {
    'bias': 'bias',
    'dark': 'dark',
    'flat': 'flat',
    'defects': 'defects',
    'ptc': 'ptc',          # needed: gains are taken from the PTC by default
}


def run_isr(detector, exposure=None, verbose=True, locate_dips=False):
    """Run a minimal ISR that preserves the sky level and the dip pixels.

    Parameters
    ----------
    detector : `int`
        Detector number.
    exposure : `int`, optional
        Exposure id. Defaults to the ``exposure_id`` built from the
        ``day_obs`` and ``seq_num`` parameters.
    verbose : `bool`, optional
        Print the configuration and calibrations used.
    locate_dips : `bool`, optional
        If `True`, invert the usual choice: enable saturation and ITL dip
        masking, with the production per-detector rule, so the pipeline's
        own masker marks the dip columns. Used by `find_dip_columns` to
        locate dips; not for measuring them.

    Returns
    -------
    exposure : `lsst.afw.image.ExposureF`
        ISR-corrected exposure, in counts. **Not** background subtracted,
        with saturated cores and dip columns left at their measured values
        unless ``locate_dips`` is set.

    Notes
    -----
    Overscan, bias, dark, flat and defects are applied; everything else is
    off, for two reasons.

    Correctness. Three ITL-specific masking steps would hide the feature:
    ``doITLDipMask`` flags the dip columns ``SUSPECT`` and ``ITL_DIP``,
    ``doITLSatSagMask`` and the edge-bleed maskers flag neighbouring
    saturation artifacts. ``doSaturation`` is off as well, so the saturated
    cores keep their real values; the dip appears in columns that contain a
    saturated star, so masking saturation removes the very thing that marks
    them. ``doInterpolate`` is off so no pixel is replaced by a fitted value.
    Note that ``doITLEdgeBleedMask`` and ``doE2VEdgeBleedMask`` must be
    turned off explicitly, because ``IsrTaskLSSTConfig.validate`` rejects
    ``doSaturation = False`` while either is on.

    Speed. The default configuration spends almost all of its time in
    brighter-fatter (``ASTIER23`` electrostatic) and deferred-charge
    correction. Neither matters for a sky-level measurement, and switching
    them off takes one detector from several minutes to about 7 seconds.
    """
    exposure = exposure_id if exposure is None else exposure
    data_id = {'instrument': 'LSSTCam', 'detector': detector,
               'exposure': exposure}

    config = IsrTaskLSSTConfig()
    config.doInterpolate = False
    if locate_dips:
        # Let the pipeline flag the dips for us, using the production rule.
        config.doSaturation = True
        config.doITLDipMask = True
        rule = copy.copy(config.overscanCamera.defaultDetectorConfig)
        rule.itlDipMinWidth = DIP_RULES[camera[detector].getName()][0]
        rule.itlDipBackgroundFraction =             DIP_RULES[camera[detector].getName()][1]
        config.overscanCamera.detectorRules[camera[detector].getName()] = rule
    else:
        # Keep every pixel the study is about.
        config.doSaturation = False
        config.doITLEdgeBleedMask = False
        config.doE2VEdgeBleedMask = False
        config.doITLDipMask = False
        config.doITLSatSagMask = False
    # Drop the expensive corrections that do not affect the sky level.
    config.doBrighterFatter = False
    config.doDeferredCharge = False
    config.doCrosstalk = False
    config.doLinearize = False
    config.doCorrectGains = False
    config.doAmpOffset = False

    raw = butler.get('raw', dataId=data_id, collections=raw_collection)
    calibs = {}
    for dataset_type, kwarg in CALIB_DATASET_TO_KWARG.items():
        calibs[kwarg] = butler.get(dataset_type, dataId=data_id,
                                   collections=calib_collection)
    calibs['camera'] = butler.get('camera', dataId=data_id,
                                  collections=calib_collection)

    if verbose:
        print(f'detector {detector} ({camera[detector].getName()}, '
              f'{camera[detector].getPhysicalType()}), exposure {exposure}')
        if locate_dips:
            print(f'  masking ON  : saturation + ITL dip, production rule')
        else:
            print(f'  masking off : saturation, ITL dip, ITL sat sag, '
                  f'edge bleed, interpolation')
        print(f'  calibs      : {sorted(k for k in calibs if k != "camera")}')

    return IsrTaskLSST(config=config).run(raw, **calibs).exposure

In [5]:
def nmad(values):
    """Normalized median absolute deviation, a robust standard deviation.

    Parameters
    ----------
    values : `numpy.ndarray`
        Input values, any units.

    Returns
    -------
    scatter : `float`
        Robust scatter, in the units of ``values``.
    """
    finite = values[np.isfinite(values)]
    return 1.4826 * np.median(np.abs(finite - np.median(finite)))


def sky_level(exposure, sample=9):
    """Robust sky level and sky noise of an ISR-corrected exposure.

    Parameters
    ----------
    exposure : `lsst.afw.image.ExposureF`
        ISR-corrected, not background subtracted, exposure.
    sample : `int`, optional
        Subsample stride, to keep this fast on a full detector.

    Returns
    -------
    sky : `float`
        Median sky level, in counts.
    noise : `float`
        Robust sky noise (nMAD), in counts. In a crowded galactic-plane
        field this is dominated by stars, not by read or photon noise.
    """
    values = masked_image(exposure)[::sample, ::sample]
    return float(np.nanmedian(values)), float(nmad(values))


def masked_image(exposure, planes=('BAD', 'NO_DATA')):
    """Image array with unusable pixels replaced by `nan`.

    ``SAT`` is deliberately **not** masked: the saturated star is what marks
    the dip columns, and ISR was run with ``doSaturation = False``.

    Parameters
    ----------
    exposure : `lsst.afw.image.ExposureF`
        ISR-corrected exposure.
    planes : `tuple` [`str`], optional
        Mask planes to blank out.

    Returns
    -------
    image : `numpy.ndarray`
        Copy of the image array, in counts, with masked pixels set to `nan`.
    """
    image = exposure.image.array.astype(float).copy()
    mask = exposure.mask
    bits = 0
    for plane in planes:
        try:
            bits |= mask.getPlaneBitMask(plane)
        except Exception:
            pass
    if bits:
        image[(mask.array & bits) != 0] = np.nan
    return image


def load_reference_stars(detector, visit=None, band=None):
    """Bright stars on one detector, from the reference catalog.

    The pipeline's own source catalog (``single_visit_star_unstandardized``)
    drops stars once they saturate -- on this exposure its brightest entry is
    16.3 mag -- so the saturated stars this study needs are exactly the ones
    missing from it. The reference catalog keeps them, and the PVI world
    coordinate system (WCS) puts them on the pixel grid.

    Parameters
    ----------
    detector : `int`
        Detector number.
    visit : `int`, optional
        Visit id. Defaults to ``exposure_id``.
    band : `str`, optional
        Filter band letter. Defaults to the band of the visit.

    Returns
    -------
    stars : `pandas.DataFrame`
        Columns ``mag`` (AB magnitude in the visit band), ``x_pix``,
        ``y_pix``, ``ra_deg`` and ``dec_deg``, brightest first.
    """
    visit = exposure_id if visit is None else visit
    data_id = {'instrument': 'LSSTCam', 'detector': detector, 'visit': visit}
    pvi = butler.get('preliminary_visit_image', dataId=data_id,
                     collections=pvi_collection)
    wcs, bbox = pvi.getWcs(), pvi.getBBox()
    if band is None:
        band = pvi.getFilter().bandLabel

    # Only the catalog shards overlapping this detector, with dimension
    # records attached -- ReferenceObjectLoader needs the shard regions.
    shards = list(butler.query_datasets(
        refcat_name, collections=refcat_collection, find_first=True,
        explain=False, with_dimension_records=True,
        data_id={'instrument': 'LSSTCam'},
        where=f'instrument=\'LSSTCam\' AND visit={visit} '
              f'AND detector={detector}'))
    loader = ReferenceObjectLoader(
        dataIds=[ref.dataId for ref in shards],
        refCats=[butler.getDeferred(ref) for ref in shards],
        name=refcat_name)

    flux_field = f'monster_ComCam_{band}'
    catalog = loader.loadPixelBox(bbox, wcs,
                                  filterName=flux_field).refCat
    flux_njy = catalog[f'{flux_field}_flux']
    # AB magnitude from flux in nJy: zero point 3631 Jy = 3.631e12 nJy.
    with np.errstate(invalid='ignore', divide='ignore'):
        mag = -2.5 * np.log10(flux_njy / 3.631e12)
    x_pix, y_pix = wcs.skyToPixelArray(catalog['coord_ra'],
                                       catalog['coord_dec'])

    stars = pd.DataFrame({
        'mag': mag, 'x_pix': x_pix, 'y_pix': y_pix,
        'ra_deg': np.degrees(catalog['coord_ra']),
        'dec_deg': np.degrees(catalog['coord_dec']),
    })
    return stars.sort_values('mag').reset_index(drop=True)


def select_stars(stars, exposure, mag_range=None, margin=None):
    """Keep stars in a magnitude range and clear of the detector edge.

    Parameters
    ----------
    stars : `pandas.DataFrame`
        Output of `load_reference_stars`.
    exposure : `lsst.afw.image.ExposureF`
        Exposure, used for its dimensions.
    mag_range : `tuple` [`float`], optional
        ``(bright, faint)`` magnitude limits. Defaults to
        ``(mag_min, mag_max)``.
    margin : `int`, optional
        Required clearance from every edge, in pixels. Defaults to
        ``edge_margin_pix``.

    Returns
    -------
    stars : `pandas.DataFrame`
        The subset, brightest first.
    """
    bright, faint = (mag_min, mag_max) if mag_range is None else mag_range
    margin = edge_margin_pix if margin is None else margin
    n_y, n_x = exposure.image.array.shape
    keep = ((stars.mag > bright) & (stars.mag < faint)
            & (stars.x_pix > margin) & (stars.x_pix < n_x - margin)
            & (stars.y_pix > margin) & (stars.y_pix < n_y - margin))
    return stars[keep].reset_index(drop=True)

In [6]:
def find_dip_columns(detector, exposure=None):
    """Locate the dip columns using the pipeline's own ITL dip masker.

    This is the ground truth for where the dips are. Guessing from star
    positions does not work: a saturated star's bleed trail is itself a
    bright column, the dip sits beside it rather than on it, and plenty of
    bright stars produce no dip at all.

    Parameters
    ----------
    detector : `int`
        Detector number. Must be one of the keys of ``DIP_RULES``.
    exposure : `int`, optional
        Exposure id. Defaults to ``exposure_id``.

    Returns
    -------
    dips : `pandas.DataFrame`
        One row per contiguous group of flagged columns, with ``x_lo``,
        ``x_hi`` and ``width_pix``; ``sat_row_max`` the highest saturated
        row in those columns and ``sat_row_fraction`` the fraction of rows
        the bleed trail saturates; widest first.

    Notes
    -----
    Runs ISR a second time with ``doSaturation`` and ``doITLDipMask`` on and
    the production rule for this detector installed, purely to read the
    ``ITL_DIP`` mask plane. The measurement itself uses the unmasked pass.
    """
    flagged = run_isr(detector, exposure=exposure, verbose=False,
                      locate_dips=True)
    mask = flagged.mask
    is_dip = (mask.array & mask.getPlaneBitMask('ITL_DIP')) != 0
    is_sat = (mask.array & mask.getPlaneBitMask('SAT')) != 0
    n_rows = is_dip.shape[0]

    columns = np.flatnonzero(is_dip.any(axis=0))
    if not columns.size:
        return pd.DataFrame(columns=['x_lo', 'x_hi', 'width_pix',
                                     'sat_row_max', 'sat_row_fraction'])

    rows_out = []
    for group in np.split(columns, np.flatnonzero(np.diff(columns) > 1) + 1):
        sat_rows = np.flatnonzero(is_sat[:, group].any(axis=1))
        rows_out.append({
            'x_lo': int(group[0]), 'x_hi': int(group[-1]),
            'width_pix': int(len(group)),
            'sat_row_max': int(sat_rows.max()) if sat_rows.size else 0,
            'sat_row_fraction': len(sat_rows) / n_rows,
        })
    dips = pd.DataFrame(rows_out).sort_values('width_pix', ascending=False)
    return dips.reset_index(drop=True)


def select_dips(dips, max_sat_fraction=None, min_width=None):
    """Keep dips with a partially filled bleed trail and enough width.

    Parameters
    ----------
    dips : `pandas.DataFrame`
        Output of `find_dip_columns`.
    max_sat_fraction : `float`, optional
        Reject dips whose trail saturates more than this fraction of the
        column. Defaults to ``max_sat_row_fraction``. A trail filling the
        whole column leaves no clean sky to measure the dip against.
    min_width : `int`, optional
        Minimum flagged width, in pixels. Defaults to ``min_dip_width_pix``.

    Returns
    -------
    dips : `pandas.DataFrame`
        The subset, widest first. Dips are dropped unless at least
        ``min_profile_rows`` rows remain beyond the trail plus its
        clearance, so every survivor has clean sky to be measured against.
    """
    max_sat_fraction = (max_sat_row_fraction if max_sat_fraction is None
                        else max_sat_fraction)
    min_width = min_dip_width_pix if min_width is None else min_width
    # Require real clean rows beyond the trail.  sat_row_fraction alone is
    # not enough: a sparse trail can still reach the last row, leaving no
    # clean band, and the profile would then be taken inside the trail.
    clean_rows = n_detector_rows - (dips.sat_row_max + trail_clearance_pix)
    keep = ((dips.sat_row_fraction < max_sat_fraction)
            & (dips.width_pix >= min_width)
            & (clean_rows >= min_profile_rows))
    return dips[keep].reset_index(drop=True)


def dip_profile(image, dip, half_width=None, clearance=None):
    """Profile across the dip columns, using rows beyond the bleed trail.

    The rows are taken past the end of the trail, where the sky is clean, so
    a trail of order 1e5 counts cannot leak into the profile. Rows are
    median combined.

    Parameters
    ----------
    image : `numpy.ndarray`
        Image array in counts, from `masked_image`.
    dip : `pandas.Series`
        One row of `find_dip_columns`.
    half_width : `int`, optional
        Half-extent in x of the profile, in pixels. Defaults to
        ``profile_half_width_pix``.
    clearance : `int`, optional
        Start this far beyond the last saturated row, in pixels. Defaults to
        ``trail_clearance_pix``.

    Returns
    -------
    x : `numpy.ndarray`
        Column coordinate, in pixels.
    profile : `numpy.ndarray`
        Median sky level at each column, in counts.
    rows : `tuple` [`int`]
        ``(y_low, y_high)`` of the band used, in pixels.
    """
    half_width = profile_half_width_pix if half_width is None else half_width
    clearance = trail_clearance_pix if clearance is None else clearance
    n_y, n_x = image.shape

    # iterrows() yields floats when a DataFrame mixes int and float columns,
    # so cast before using these as slice indices.
    x_center = (int(dip.x_lo) + int(dip.x_hi)) // 2
    x_lo = int(max(0, x_center - half_width))
    x_hi = int(min(n_x, x_center + half_width))

    y_lo = int(dip.sat_row_max) + clearance
    if y_lo > n_y - min_profile_rows:
        raise ValueError(
            f'dip at x {int(dip.x_lo)}-{int(dip.x_hi)} has only '
            f'{n_y - y_lo} clean rows beyond its trail, fewer than the '
            f'{min_profile_rows} required; select_dips should have dropped '
            f'it')
    band = image[y_lo:n_y, x_lo:x_hi]
    with warnings.catch_warnings():
        warnings.simplefilter('ignore', RuntimeWarning)
        profile = np.nanmedian(band, axis=0)
    return np.arange(x_lo, x_hi), profile, (y_lo, n_y)


def measure_dip(x, profile, dip, ref_width=60):
    """Depth of the dip below the surrounding columns.

    Parameters
    ----------
    x : `numpy.ndarray`
        Column coordinate from `dip_profile`, in pixels.
    profile : `numpy.ndarray`
        Profile from `dip_profile`, in counts.
    dip : `pandas.Series`
        One row of `find_dip_columns`, giving the flagged column range.
    ref_width : `int`, optional
        Width of the reference region at each end of the profile, in pixels.

    Returns
    -------
    depth : `float`
        Reference level minus the minimum inside the flagged columns, in
        counts. Positive means the dip columns sit below their surroundings.
    reference : `float`
        Reference sky level from the ends of the profile, in counts.
    """
    reference = float(np.nanmedian(np.r_[profile[:ref_width],
                                         profile[-ref_width:]]))
    inside = (x >= dip.x_lo) & (x <= dip.x_hi)
    if not inside.any() or np.all(~np.isfinite(profile[inside])):
        return np.nan, reference
    return reference - float(np.nanmin(profile[inside])), reference

In [7]:
def show_sky_stretch(exposure, ax=None, low=3.0, high=2.0, zscale=False,
                     title=None):
    """Display an exposure with the stretch set around the sky level.

    A dip of a few tens of counts is invisible at a normal stretch. The
    default window is asymmetric -- further below the sky than above it --
    because a crowded field has a long positive tail from the stars while
    the feature of interest is a deficit.

    Parameters
    ----------
    exposure : `lsst.afw.image.ExposureF`
        ISR-corrected, not background subtracted, exposure.
    ax : `matplotlib.axes.Axes`, optional
        Axes to draw into. A new figure is made if omitted.
    low, high : `float`, optional
        Extent of the stretch below and above the sky level, in units of the
        sky scatter. Asymmetric because the feature is a deficit while a
        star field has a long positive tail. Clipping much tighter than the
        scatter turns the image into salt and pepper.
    zscale : `bool`, optional
        Use `~astropy.visualization.ZScaleInterval` instead.
    title : `str`, optional
        Axes title.

    Returns
    -------
    ax : `matplotlib.axes.Axes`
        The axes drawn into.
    """
    image = masked_image(exposure)
    sky, noise = sky_level(exposure)
    if zscale:
        finite = image[np.isfinite(image)]
        vmin, vmax = ZScaleInterval().get_limits(finite[::7])
        if not vmin < vmax:
            vmin, vmax = sky - low * noise, sky + high * noise
    else:
        vmin, vmax = sky - low * noise, sky + high * noise

    if ax is None:
        _, ax = plt.subplots(figsize=(9, 9))
    mappable = ax.imshow(image, vmin=vmin, vmax=vmax, origin='lower',
                         interpolation='nearest', aspect='equal', cmap='gray')
    ax.set_xlabel('x (pixels, across columns)')
    ax.set_ylabel('y (pixels, along columns)')
    ax.set_title(title or f'{vmin:.0f} to {vmax:.0f} counts')
    plt.colorbar(mappable, ax=ax, label='counts', fraction=0.046, pad=0.04)
    return ax


def mask_plane_census(exposure, planes=('SAT', 'ITL_DIP', 'SUSPECT', 'BAD',
                                        'INTRP')):
    """Count pixels set in each mask plane.

    Parameters
    ----------
    exposure : `lsst.afw.image.ExposureF`
        Exposure to examine.
    planes : `tuple` [`str`], optional
        Mask plane names to count.

    Returns
    -------
    counts : `dict` [`str`, `int`]
        Pixels set in each plane present in the mask.
    """
    mask = exposure.mask
    counts = {}
    for plane in planes:
        try:
            bit = mask.getPlaneBitMask(plane)
        except Exception:
            continue
        counts[plane] = int(((mask.array & bit) != 0).sum())
    return counts

<a id='visits'></a>
## 4. Find low galactic latitude visits

Saturated stars are plentiful toward the galactic plane. The survey has two dedicated plane programs, `bulgy` and `dusty_plane`, which give hundreds of visits per night within a degree of `b = 0`.

In [8]:
visits = find_low_galactic_latitude_visits(
    search_day_obs_min, search_day_obs_max,
    max_abs_b_deg=max_abs_gal_b_deg,
    targets=target_names,
    min_exp_time=min_exp_time_s)

print(f'n visits with |b| < {max_abs_gal_b_deg:g} deg: {len(visits)}')
print(f'galactic latitude range: {visits.gal_b.min():+.3f} to '
      f'{visits.gal_b.max():+.3f} deg')

columns = ['visit_id', 'day_obs', 'seq_num', 'band', 'exp_time',
           'gal_l', 'gal_b', 'airmass', 'target_name']
visits[columns].head(15)

n visits with |b| < 90 deg: 985
galactic latitude range: -80.670 to +53.332 deg


,visit_id,day_obs,seq_num,band,exp_time,gal_l,gal_b,airmass,target_name
62,2026070800084,20260708,84,r,30.0,354.195163,-10.231760,1.017846,lowdust
12,2026070800034,20260708,34,i,30.0,354.195130,-10.231838,1.016644,lowdust
98,2026070800120,20260708,120,r,30.0,343.464490,-10.654231,1.092903,lowdust
45,2026070800067,20260708,67,i,30.0,343.464477,-10.654241,1.060742,lowdust
191,2026070800213,20260708,213,i,30.0,357.193974,-10.754468,1.120047,lowdust
139,2026070800161,20260708,161,r,30.0,357.194020,-10.754504,1.058466,lowdust
140,2026070800162,20260708,162,r,30.0,0.202360,-11.212285,1.046670,lowdust
192,2026070800214,20260708,214,i,30.0,0.202391,-11.212380,1.104446,lowdust
46,2026070800068,20260708,68,i,30.0,346.479250,-11.319105,1.043826,lowdust
99,2026070800121,20260708,121,r,30.0,346.479291,-11.319149,1.070175,lowdust


In [9]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(visits.gal_b, bins=40, color='C0')
axes[0].set_xlabel('galactic latitude b (deg)')
axes[0].set_ylabel('number of visits')
axes[0].set_title(f'{len(visits)} plane visits, '
                  f'day_obs {search_day_obs_min}-{search_day_obs_max}')

axes[1].scatter(visits.gal_l, visits.gal_b, s=6, alpha=0.5)
axes[1].set_xlabel('galactic longitude l (deg)')
axes[1].set_ylabel('galactic latitude b (deg)')
axes[1].set_title('field coverage')

plt.tight_layout()
pdf.savefig(fig, bbox_inches='tight')
plt.show()

# The exposure studied below.
chosen = visits[(visits.day_obs == day_obs) & (visits.seq_num == seq_num)]
if len(chosen):
    row = chosen.iloc[0]
    print(f'studying exposure {exposure_id}: {row.target_name}, '
          f'band {row.band}, exp_time {row.exp_time:g} s, '
          f'b = {row.gal_b:+.3f} deg, airmass {row.airmass:.3f}')
else:
    print(f'studying exposure {exposure_id} (not in the filtered visit list)')

studying exposure 2026070800022: lowdust, band i, exp_time 30 s, b = -14.277 deg, airmass 1.052


/lscratch/roodman/ipykernel_3660396/3838454396.py:16: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


<a id='isr'></a>
## 5. Run ISR without dip or saturation masking

About 7 seconds per detector with the minimal configuration. The result is cached in `itl_exposure` and reused below.

In [10]:
itl_exposure = run_isr(itl_detector)
itl_image = masked_image(itl_exposure)
itl_sky, itl_noise = sky_level(itl_exposure)

print()
print(f'sky level   = {itl_sky:.1f} counts')
print(f'sky scatter = {itl_noise:.1f} counts (nMAD over the detector; '
      f'includes stars, so larger than the pixel noise)')
print(f'peak pixel  = {np.nanmax(itl_image):.0f} counts')
print(f'mask planes : {mask_plane_census(itl_exposure)}')

INFO:lsst.isrLSST:Seeding dithering random number generator with 3355448846.


detector 14 (R02_S12, ITL), exposure 2026070800022
  masking off : saturation, ITL dip, ITL sat sag, edge bleed, interpolation
  calibs      : ['bias', 'dark', 'defects', 'flat', 'ptc']


INFO:lsst.isrLSST:Using gain values to convert from adu to electron units.


INFO:lsst.isrLSST:Assembling CCD from amplifiers.


INFO:lsst.isrLSST:Applying bias correction.


...

INFO:lsst.isrLSST:Masking non-finite (NAN, inf) value pixels.


INFO:lsst.isrLSST:Applying flat correction.


INFO:lsst.isrLSST:Setting values in large contiguous bad regions.


INFO:lsst.isrLSST:Set 121145 BAD pixels to 1733.459473.



sky level   = 1733.3 counts
sky scatter = 61.2 counts (nMAD over the detector; includes stars, so larger than the pixel noise)
peak pixel  = 245563 counts
mask planes : {'SAT': 0, 'ITL_DIP': 0, 'SUSPECT': 59075, 'BAD': 121145, 'INTRP': 0}


Three things to check in that output:

- A sky level of order 10^3 counts, so the image is **not** background
  subtracted and the dip has a reference level to sit below.
- A peak pixel of order 10^5 counts, so bright stars are saturated and kept
  at their measured values.
- `SAT` and `ITL_DIP` both zero, because no masking was applied.

<a id='display'></a>
## 6. Locate the dips

The pipeline's own masker says where the dips are. Dips whose bleed trail fills the whole column are dropped: there is no clean sky left to measure them against.

In [11]:
dips_all = find_dip_columns(itl_detector)
dips = select_dips(dips_all)

print(f'n dips flagged by the pipeline      : {len(dips_all)}')
print(f'n with a partial trail, width >= {min_dip_width_pix} px : {len(dips)}')
print()
print('sat_row_fraction is the fraction of the column the trail saturates.')
dips_all

INFO:lsst.isrLSST:Seeding dithering random number generator with 3355448846.


INFO:lsst.isrLSST:Using gain values to convert from adu to electron units.


INFO:lsst.isrLSST:Assembling CCD from amplifiers.


INFO:lsst.isrLSST:Found ITL dip (width 6; bkg 1711.51); masking column 2361 to 2370.


INFO:lsst.isrLSST:Found ITL dip (width 6; bkg 1711.51); masking column 486 to 495.


...

INFO:lsst.isrLSST:Applying flat correction.


INFO:lsst.isrLSST:Setting values in large contiguous bad regions.


INFO:lsst.isrLSST:Set 121145 BAD pixels to 1733.459717.


n dips flagged by the pipeline      : 9
n with a partial trail, width >= 10 px : 6

sat_row_fraction is the fraction of the column the trail saturates.


,x_lo,x_hi,width_pix,sat_row_max,sat_row_fraction
0,3196,3242,47,3999,1.00000
1,892,910,19,2522,0.26125
2,2318,2333,16,3999,0.10175
3,604,619,16,3292,0.18500
4,3115,3129,15,3200,0.16600
5,3552,3565,14,3067,0.17625
6,2134,2146,13,3210,0.08250
7,486,495,10,1793,0.05075
8,2361,2370,10,3999,0.03275


In [12]:
# Which star produced each dip?  The reference catalog keeps the saturated
# stars that the pipeline's own source catalog drops.
ref_stars = load_reference_stars(itl_detector)
rows = []
for _, dip in dips.iterrows():
    x_center = 0.5 * (dip.x_lo + dip.x_hi)
    near = ref_stars[np.abs(ref_stars.x_pix - x_center) < 0.5 * dip.width_pix + 15]
    brightest = near.mag.min() if len(near) else np.nan
    rows.append({'x_lo': dip.x_lo, 'x_hi': dip.x_hi,
                 'width_pix': dip.width_pix,
                 'sat_row_fraction': round(dip.sat_row_fraction, 3),
                 'brightest_star_mag': brightest})
print('Dips and the brightest reference star in their columns:')
print(pd.DataFrame(rows).to_string(index=False))

INFO:lsst.meas.algorithms.loadReferenceObjects.ReferenceObjectLoader:Loading reference objects from the_monster_20250219 in region bounded by [276.79150621, 277.28126220], [-46.33327862, -45.99144026] RA Dec


INFO:lsst.meas.algorithms.loadReferenceObjects.ReferenceObjectLoader:Loaded 6885 reference objects


Dips and the brightest reference star in their columns:
  x_lo   x_hi  width_pix  sat_row_fraction  brightest_star_mag
 892.0  910.0       19.0             0.261            8.999254
 604.0  619.0       16.0             0.185           12.573283
3115.0 3129.0       15.0             0.166            9.759306
3552.0 3565.0       14.0             0.176            9.669573
2134.0 2146.0       13.0             0.082           10.522155
 486.0  495.0       10.0             0.051           11.079594


In [13]:
fig, axes = plt.subplots(1, 2, figsize=(16, 7.5))

show_sky_stretch(itl_exposure, ax=axes[0], zscale=True,
                 title=f'{camera[itl_detector].getName()}: zscale stretch')

image = itl_image
mappable = axes[1].imshow(image,
                          vmin=itl_sky - 3 * itl_noise,
                          vmax=itl_sky + 2 * itl_noise,
                          origin='lower', interpolation='nearest',
                          aspect='equal', cmap='gray')
axes[1].set_title(f'sky-level stretch, sky = {itl_sky:.0f} counts')
axes[1].set_xlabel('x (pixels, across columns)')
axes[1].set_ylabel('y (pixels, along columns)')
plt.colorbar(mappable, ax=axes[1], label='counts', fraction=0.046, pad=0.04)

for _, dip in dips.iterrows():
    axes[1].axvline(dip.x_lo, color='C1', lw=0.5, alpha=0.8)
    axes[1].axvline(dip.x_hi, color='C1', lw=0.5, alpha=0.8)

plt.tight_layout()
pdf.savefig(fig, bbox_inches='tight')
plt.show()

/lscratch/roodman/ipykernel_3660396/1368080201.py:23: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


<a id='cuts'></a>
## 7. The dip against the sky

Top row: the dip columns, stretched around the sky level, with the flagged range and the end of the bleed trail marked. Bottom row: the profile across columns, taken from rows beyond the trail where the sky is clean.

In [14]:
def plot_dips(image, dips, sky, noise, label, n_dips=None):
    """Plot each dip and its cross-column profile.

    Parameters
    ----------
    image : `numpy.ndarray`
        Image array in counts, from `masked_image`.
    dips : `pandas.DataFrame`
        Output of `select_dips`.
    sky, noise : `float`
        Sky level and sky scatter, in counts.
    label : `str`
        Detector name, used in titles and the output filename.
    n_dips : `int`, optional
        Number of dips to plot. Defaults to ``n_dips_to_plot``.

    Returns
    -------
    measurements : `pandas.DataFrame`
        Per-dip depth, in counts and as a fraction of the sky level
        (dimensionless, depth over sky).
    """
    n_dips = n_dips_to_plot if n_dips is None else n_dips
    selected = dips.head(n_dips)
    if not len(selected):
        print(f'{label}: no dips with a partial bleed trail')
        return pd.DataFrame()

    fig, axes = plt.subplots(2, len(selected),
                             figsize=(5.4 * len(selected), 9.5),
                             squeeze=False)
    rows_out = []
    for i, (_, dip) in enumerate(selected.iterrows()):
        x, profile, (y_lo, y_hi) = dip_profile(image, dip)
        depth, reference = measure_dip(x, profile, dip)

        mappable = axes[0, i].imshow(
            image[:, int(x[0]):int(x[-1]) + 1],
            vmin=sky - 3 * noise, vmax=sky + 2 * noise,
            origin='lower', extent=(x[0], x[-1] + 1, 0, image.shape[0]),
            aspect='auto', cmap='gray', interpolation='nearest')
        axes[0, i].axvline(dip.x_lo, color='C1', lw=0.8)
        axes[0, i].axvline(dip.x_hi, color='C1', lw=0.8)
        axes[0, i].axhline(dip.sat_row_max, color='C3', lw=0.8, ls='--')
        axes[0, i].axhspan(y_lo, y_hi, color='C0', alpha=0.12)
        axes[0, i].set_title(f'{label}  x {dip.x_lo}-{dip.x_hi}\n'
                             f'trail ends y = {dip.sat_row_max}, '
                             f'{dip.sat_row_fraction:.0%} of column')
        axes[0, i].set_xlabel('x (pixels)')
        if i == 0:
            axes[0, i].set_ylabel('y (pixels, along columns)')
        plt.colorbar(mappable, ax=axes[0, i], label='counts',
                     fraction=0.046, pad=0.04)

        axes[1, i].plot(x, profile, lw=0.9, color='C0')
        axes[1, i].axhline(reference, color='C2', lw=1.0,
                           label=f'reference {reference:.1f} counts')
        axes[1, i].axvspan(dip.x_lo, dip.x_hi, color='C1', alpha=0.2,
                           label='flagged ITL_DIP')
        axes[1, i].set_xlabel('x (pixels, across columns)')
        if i == 0:
            axes[1, i].set_ylabel('median sky (counts)')
        axes[1, i].legend(fontsize=7, loc='lower left')
        axes[1, i].set_title(f'rows {y_lo}-{y_hi}, beyond the trail\n'
                             f'depth {depth:.1f} counts '
                             f'({depth / sky:.2e} of sky)')

        rows_out.append({'x_lo': dip.x_lo, 'x_hi': dip.x_hi,
                         'width_pix': dip.width_pix,
                         'sat_row_fraction': dip.sat_row_fraction,
                         'depth_counts': depth,
                         'depth_over_sky': depth / sky,
                         'reference_counts': reference})

    plt.tight_layout()
    pdf.savefig(fig, bbox_inches='tight')
    plt.show()
    return pd.DataFrame(rows_out)


itl_measurements = plot_dips(itl_image, dips, itl_sky, itl_noise,
                             camera[itl_detector].getName())
itl_measurements

/lscratch/roodman/ipykernel_3660396/4234419187.py:77: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


,x_lo,x_hi,width_pix,sat_row_fraction,depth_counts,depth_over_sky,reference_counts
0,892.0,910.0,19.0,0.26125,72.251770,0.041685,1736.829651
1,604.0,619.0,16.0,0.18500,68.760132,0.039671,1763.920898
2,3115.0,3129.0,15.0,0.16600,67.559082,0.038978,1682.828979


<a id='e2v'></a>
## 8. E2V control

No E2V detector carries a dip rule in the pipeline configuration, so there is nothing for the masker to find. The check is that the same procedure applied to an E2V CCD in the same exposure turns up no dip, and that its saturated stars leave the sky beside them flat.

In [15]:
e2v_exposure = run_isr(e2v_detector)
e2v_image = masked_image(e2v_exposure)
e2v_sky, e2v_noise = sky_level(e2v_exposure)
print()
print(f'sky level = {e2v_sky:.1f} counts, scatter = {e2v_noise:.1f} counts')
print(f'E2V detector in DIP_RULES: '
      f'{camera[e2v_detector].getName() in DIP_RULES}')

# Profile across the columns of the brightest star on the E2V CCD, using the
# same geometry as the ITL dips.
e2v_stars = load_reference_stars(e2v_detector)
brightest = e2v_stars.iloc[0]
e2v_dip = pd.Series({'x_lo': int(brightest.x_pix) - 8,
                     'x_hi': int(brightest.x_pix) + 8,
                     'width_pix': 17,
                     'sat_row_max': int(brightest.y_pix) + 150,
                     'sat_row_fraction': 0.0})
x, profile, (y_lo, y_hi) = dip_profile(e2v_image, e2v_dip)
depth, reference = measure_dip(x, profile, e2v_dip)
print(f'brightest E2V star: {brightest.mag:.2f} mag at '
      f'x = {brightest.x_pix:.0f}, y = {brightest.y_pix:.0f}')
print(f'deficit beside its columns: {depth:.1f} counts '
      f'({depth / e2v_sky:.2e} of sky)')

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
mappable = axes[0].imshow(e2v_image[:, int(x[0]):int(x[-1]) + 1],
                          vmin=e2v_sky - 3 * e2v_noise,
                          vmax=e2v_sky + 2 * e2v_noise,
                          origin='lower',
                          extent=(x[0], x[-1] + 1, 0, e2v_image.shape[0]),
                          aspect='auto', cmap='gray', interpolation='nearest')
axes[0].set_title(f'{camera[e2v_detector].getName()} (E2V), '
                  f'{brightest.mag:.2f} mag star')
axes[0].set_xlabel('x (pixels)')
axes[0].set_ylabel('y (pixels, along columns)')
plt.colorbar(mappable, ax=axes[0], label='counts', fraction=0.046, pad=0.04)

axes[1].plot(x, profile, lw=0.9)
axes[1].axhline(reference, color='C2', lw=1.0,
                label=f'reference {reference:.1f} counts')
axes[1].axvspan(e2v_dip.x_lo, e2v_dip.x_hi, color='C1', alpha=0.2,
                label='star columns')
axes[1].set_xlabel('x (pixels, across columns)')
axes[1].set_ylabel('median sky (counts)')
axes[1].set_title(f'deficit {depth:.1f} counts '
                  f'({depth / e2v_sky:.2e} of sky)')
axes[1].legend(fontsize=8)
plt.tight_layout()
pdf.savefig(fig, bbox_inches='tight')
plt.show()

INFO:lsst.isrLSST:Seeding dithering random number generator with 3355448871.


detector 39 (R11_S10, E2V), exposure 2026070800022
  masking off : saturation, ITL dip, ITL sat sag, edge bleed, interpolation
  calibs      : ['bias', 'dark', 'defects', 'flat', 'ptc']


INFO:lsst.isrLSST:Using gain values to convert from adu to electron units.


INFO:lsst.isrLSST:Assembling CCD from amplifiers.


INFO:lsst.isrLSST:Applying bias correction.


...


sky level = 1673.4 counts, scatter = 50.6 counts
E2V detector in DIP_RULES: False


INFO:lsst.meas.algorithms.loadReferenceObjects.ReferenceObjectLoader:Loading reference objects from the_monster_20250219 in region bounded by [275.25341344, 275.73348520], [-45.48518404, -45.14492861] RA Dec


INFO:lsst.meas.algorithms.loadReferenceObjects.ReferenceObjectLoader:Loaded 8516 reference objects


brightest E2V star: 10.16 mag at x = 1500, y = 1212
deficit beside its columns: 1.3 counts (7.73e-04 of sky)


/lscratch/roodman/ipykernel_3660396/3921379982.py:50: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [16]:
if len(itl_measurements):
    print('ITL dip depths on '
          f'{camera[itl_detector].getName()} versus the E2V control')
    print('(depth_over_sky is dimensionless: deficit in counts over sky in '
          'counts)')
    summary = pd.DataFrame({
        'case': [f'ITL {camera[itl_detector].getName()} (median of '
                 f'{len(itl_measurements)} dips)',
                 f'E2V {camera[e2v_detector].getName()} '
                 f'({brightest.mag:.1f} mag star)'],
        'depth_counts': [itl_measurements.depth_counts.median(), depth],
        'depth_over_sky': [itl_measurements.depth_over_sky.median(),
                           depth / e2v_sky],
        'sky_counts': [itl_sky, e2v_sky],
    })
    print(summary.to_string(index=False))

ITL dip depths on R02_S12 versus the E2V control
(depth_over_sky is dimensionless: deficit in counts over sky in counts)
                          case  depth_counts  depth_over_sky  sky_counts
ITL R02_S12 (median of 3 dips)     68.760132        0.039671 1733.277344
   E2V R11_S10 (10.2 mag star)      1.293640        0.000773 1673.383606


<a id='pvi'></a>
## 9. Why the PVI cannot be used directly

The Preliminary Visit Image is background subtracted, so the sky the dip sits in has already been removed. Its background model is a separate dataset.

In [17]:
pvi_data_id = {'instrument': 'LSSTCam', 'detector': itl_detector,
               'visit': exposure_id}
pvi = butler.get('preliminary_visit_image', dataId=pvi_data_id,
                 collections=pvi_collection)
background = butler.get('preliminary_visit_image_background',
                        dataId=pvi_data_id, collections=pvi_collection)

pvi_median = float(np.nanmedian(pvi.image.array))
bkgd_median = float(np.nanmedian(background.getImage().array))
print(f'PVI median            = {pvi_median:8.1f} counts '
      f'(background subtracted)')
print(f'background median     = {bkgd_median:8.1f} counts')
print(f'sum                   = {pvi_median + bkgd_median:8.1f} counts')
print(f'minimal ISR sky       = {itl_sky:8.1f} counts')
print()
print('The PVI sky sits near zero, so a dip of a few tens of counts has no '
      'reference level without adding the background back.  The PVI is still '
      'what provides the WCS and the photometric calibration used in '
      'section 6.')

PVI median            =      9.7 counts (background subtracted)
background median     =   1716.8 counts
sum                   =   1726.5 counts
minimal ISR sky       =   1733.3 counts

The PVI sky sits near zero, so a dip of a few tens of counts has no reference level without adding the background back.  The PVI is still what provides the WCS and the photometric calibration used in section 6.


<a id='save'></a>
## 10. Close the figure file

In [18]:
pdf.close()
print(f'wrote {pdf_path}')
print(f'pages: visit selection, detector overview, ITL dip profiles, '
      f'E2V control')

wrote /sdf/data/rubin/user/roodman/LSST/rubin-work/studies/camera/itl_dip/itl_dip_20260708_22.pdf
pages: visit selection, detector overview, ITL dip profiles, E2V control


## Summary

How to see the ITL dip, in the order the choices matter:

1. **Use a sparse field.** A `lowdust` field shows the sky clearly between
   stars. Crowded galactic-plane fields (`bulgy`, `dusty_plane`) have more
   saturated stars but their confusion noise is as large as the dip, which
   makes the depth unmeasurable.
2. **Run ISR yourself with the masking off.** `doITLDipMask`,
   `doSaturation`, `doITLSatSagMask` and the edge-bleed maskers all remove
   pixels this study needs; `doInterpolate` replaces them with fitted
   values. Turning off brighter-fatter and deferred charge takes one
   detector from minutes to about 7 seconds and changes nothing about the
   sky.
3. **Find the dips with the pipeline's masker, not from star positions.** A
   saturated star's bleed trail is itself a bright column, the dip sits
   beside it, and many bright stars produce no dip. `find_dip_columns`
   installs the production rule and reads the `ITL_DIP` plane.
4. **Prefer a partially filled bleed trail.** A trail spanning the whole
   column leaves no clean sky to reference. Profiling rows beyond the end of
   the trail also keeps the trail's ~1e5 counts out of the measurement.
5. **Set the stretch around the sky and add a colorbar.** A zscale stretch
   hides the dip entirely.

On the example exposure the dips measure a few tens of counts deep against a
sky of order 1600 counts, a few percent of sky, and the same procedure on an
E2V CCD finds nothing comparable.

To extend: loop over the 25 detectors in `DIP_RULES` and correlate depth
against the star magnitude, the amp, and the dip width; or compare measured
depths against the `itlDipBackgroundFraction` the production configuration
assigns each detector.